# Setup

In [1]:
# user setup
import os

# custom Kaggle login (environment variables are recommended)
os.environ["KAGGLE_USERNAME"] = "" or os.getenv("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = "" or os.getenv("KAGGLE_KEY")

# spark configuration
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64" or os.getenv("JAVA_HOME")
os.environ["SPARK_HOME"] = "" or os.getenv("SPARK_HOME")

# configuration
MIN_KW_FREQ = 5 # minimum frequency of keywords to be included in the profile (for the dictionary approach)
FEATURES_SIZE = 5000 # number of features, buckets of the hash (for the hashing approach)
LSH_NUM_SKETCHES = 100 # number of random hyperplanes (signatures/sketches)
LSH_BANDS = 10 # TODO: play with b and r (calculate the threshold for different values)
LSH_ROWS = 10
LSH_MAX_BUCKET_SIZE = 500 # maximum number of users/articles in the same bucket
RECOMMENDED_ARTICLES = 10 # number of articles to recommend to each user
assert LSH_NUM_SKETCHES == LSH_BANDS * LSH_ROWS, "invalid LSH_NUM_SKETCHES, must be equal to LSH_BANDS * LSH_ROWS"


In [2]:
# setup dataset
%pip install kaggle
!test -d dataset && echo "Skipping dataset download" || (kaggle datasets download -d "benjaminawd/new-york-times-articles-comments-2020" && unzip -d dataset new-york-times-articles-comments-2020.zip && rm -f new-york-times-articles-comments-2020.zip 2> /dev/null)


Note: you may need to restart the kernel to use updated packages.
Skipping dataset download


In [3]:
# setup spark
!test -d spark && echo "Skipping spark download" || (wget -q https://dlcdn.apache.org/spark/spark-4.1.1/spark-4.1.1-bin-hadoop3.tgz -O spark.tgz && mkdir spark && tar xvf spark.tgz -C spark --strip-components=1 > /dev/null && rm spark.tgz)
%pip install pyspark
%pip install py4j
import pyspark
ss = (pyspark.sql.SparkSession
    .builder
    .getOrCreate())
sc = ss.sparkContext


Skipping spark download
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
csv_options = {
    "header": "true",
    "inferSchema": "true",
    "quote": '"',
    "escape": '"', # quotes " in the dataset are escaped as ""
    "multiLine": "true",
    "mode": "PERMISSIVE",
}

full_articles = ss.read.options(**csv_options).csv("dataset/nyt-articles-2020.csv")
full_comments = ss.read.options(**csv_options).csv("dataset/nyt-comments-part0.csv") # TODO: this is a subset of all comments

print(f"Number of articles: {full_articles.count()}")
print(f"Number of comments: {full_comments.count()}")
full_articles.take(1), full_comments.take(1)


Number of articles: 16787
Number of comments: 500000


([Row(newsdesk='Editorial', section='Opinion', subsection=None, material='Editorial', headline='Protect Veterans From Fraud', abstract='Congress could do much more to protect Americans who have served their country from predatory for-profit colleges.', keywords="['Veterans', 'For-Profit Schools', 'Financial Aid (Education)', 'Frauds and Swindling', 'Colleges and Universities', 'Veterans Affairs Department', 'Federal Trade Commission', 'University of Phoenix', 'Career Education Corporation']", word_count=680, pub_date=datetime.datetime(2020, 1, 1, 0, 18, 54), n_comments=186, uniqueID='nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')],
 [Row(commentID=104387472, status='approved', commentSequence=104387472, userID=60215558, userDisplayName='magicisnotreal', userLocation='earth', userTitle=None, commentBody='Here is something I think is fraudulent that vets are subject to\n If you use your VA home loan option you have to pay higher interest rates regardless of your credit rating becua

In [5]:
%pip install mmh3 # non-cryptographic hash, better distribution
# utility hash function: bytes | str -> signed int 32
def h(data):
    import mmh3
    return mmh3.hash(data)


Note: you may need to restart the kernel to use updated packages.


## RDD

In [6]:
# parse keywords column utility
def parse_keywords(row):
    import re
    if not row["keywords"]: return []
    kws = re.sub(r'\[|\]|\'|\"', '', row["keywords"].lower()).strip()
    if not kws: return []
    return [w.strip() for w in kws.split(",") if len(w.strip()) > 0]

# keep only relevant information for recommendations porpouse, convert from dataframe to RDD tuples
articles = (full_articles
            .select("uniqueID", "newsdesk", "section", "subsection", "keywords")
            .rdd
            .map(lambda row: (row["uniqueID"], row["newsdesk"], row["section"], row["subsection"], parse_keywords(row))))
comments = (full_comments
            .select("articleID", "userID")
            .rdd
            .map(lambda row: (row["articleID"], row["userID"])))
users = comments.map(lambda x: x[1]).distinct()

print(f"Number of articles: {articles.count()}")
print(f"Number of comments: {comments.count()}")
print(f"Number of users: {users.count()}")
articles.take(1), comments.take(1)


Number of articles: 16787
Number of comments: 500000
Number of users: 91437


([('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd',
   'Editorial',
   'Opinion',
   None,
   ['veterans',
    'for-profit schools',
    'financial aid (education)',
    'frauds and swindling',
    'colleges and universities',
    'veterans affairs department',
    'federal trade commission',
    'university of phoenix',
    'career education corporation'])],
 [('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 60215558)])

# Building Profiles

## Utility Matrix

In [7]:
utility_matrix = comments.distinct() # ignore multiple comments from same user on same article
print(f"Number of entries in sparse utility matrix: {utility_matrix.count()}")
utility_matrix.take(5)


Number of entries in sparse utility matrix: 356283


[('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 60215558),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 65691034),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 65110053),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 66232009),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 73299044)]

## Articles Profile - via dictionary

In [8]:
# count frequencies
keywords = (articles
            .flatMap(lambda row: [(k, 1) for k in row[4]])
            .reduceByKey(lambda a, b: a + b))

# prune too rare keywords
pruned_keywords = (keywords
                   .filter(lambda x: x[1] >= MIN_KW_FREQ)
                   .map(lambda x: f"KEY {x[0]}")
                   .collect())

# extract all features
features = pruned_keywords + (articles
                              .flatMap(lambda row: [f"{f} {row[i+1].lower().strip()}" for i, f in enumerate(["NEW", "SEC", "SUB"]) if row[i+1]])
                              .distinct()
                              .collect())

# features mapping dictionary, needs to be broadcasted to every node
features_mapping = {f: i for i, f in enumerate(features)}
print(f"Extacted {len(features_mapping)} features (broadcasting ~{((100 * 4) + 4) * len(features_mapping) // 1024} KB)")
broadcast_dict = sc.broadcast(features_mapping)

# build sparse vectors for profiles (article_id, feature_index)
def build_article_profile_dict(row):
    mapping = broadcast_dict.value
    res = []
    for i, f in enumerate(["NEW", "SEC", "SUB"]):
        if not row[i+1]: continue
        f = f"{f} {row[i+1].lower().strip()}"
        if f not in mapping: continue
        res.append((row[0], mapping[f]))
    for k in row[4]:
        k = f"KEY {k.lower().strip()}"
        if k not in mapping: continue
        res.append((row[0], mapping[k]))
    return res

articles_profile_dict = (articles
                         .flatMap(build_article_profile_dict) # (article_id, feature_index)
                         .map(lambda x: (x[0], (x[1], 1)))) # (article_id, (feature_index, frequency = 1))
print(f"Number of entries in sparse articles profile via dictionary: {articles_profile_dict.count()}")
articles_profile_dict.take(10)


Extacted 3555 features (broadcasting ~1402 KB)
Number of entries in sparse articles profile via dictionary: 158079


[('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (3384, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (3385, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (0, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (1, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (2, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (3, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (4, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (5, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (6, 1)),
 ('nyt://article/9edddb54-0aa3-5835-a833-d311a76f1e7c', (3386, 1))]

## Articles Profile - via hashing

In [9]:
# build sparse vectors for profiles (article_id, feature_hash)
def build_article_profile_hash(row):
    res = []
    for i, f in enumerate(["NEW", "SEC", "SUB"]):
        if not row[i+1]: continue
        f = f"{f} {row[i+1].lower().strip()}"
        res.append((row[0], h(f) % FEATURES_SIZE))
    for k in row[4]:
        k = f"KEY {k.lower().strip()}"
        res.append((row[0], h(k) % FEATURES_SIZE))
    return res

articles_profile_hash = (articles
                         .flatMap(build_article_profile_hash) # (article_id, feature_hash)
                         .map(lambda x: (x[0], (x[1], 1)))) # (article_id, (feature_hash, frequency = 1))
print(f"Number of entries in sparse articles profile via hash: {articles_profile_hash.count()}")
articles_profile_hash.take(10)


Number of entries in sparse articles profile via hash: 185396


[('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (653, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (2704, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (2692, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (1483, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (4497, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (260, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (1675, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (2683, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (3196, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (598, 1))]

## WARNING

From now on, the operations will be **independent** of how the profiles were computed (via dictionary or hashing).
Change the content of next cell to change approach: (`articles_profile_dict` or `articles_profile_hash`)

In [10]:
# articles_profile = articles_profile_dict
articles_profile = articles_profile_hash


## Users Profile

In [11]:
# build user profiles by joining utility matrix with article profiles, then counting feature frequencies for each user
users_profile = (utility_matrix
                 .join(articles_profile) # (article_id, (user_id, (feature_index, frequency = 1)))
                 .map(lambda x: ((x[1][0], x[1][1][0]), 1)) # ((user_id, feature_index), 1)
                 .reduceByKey(lambda a, b: a + b) # ((user_id, feature_index), frequency)
                 .map(lambda x: (x[0][0], (x[0][1], x[1])))) # (user_id, (feature_index, frequency))
print(f"Number of entries in sparse users profile: {users_profile.count()}")
users_profile.take(10)


Number of entries in sparse users profile: 2801463


[(5646097, (535, 36)),
 (5646097, (2881, 36)),
 (5646097, (4221, 36)),
 (86273619, (535, 11)),
 (86273619, (2881, 11)),
 (86273619, (4221, 11)),
 (61536727, (535, 9)),
 (61536727, (2881, 9)),
 (61536727, (4221, 9)),
 (41508509, (535, 20))]

# Computing Similarity

## Profiles sketches (signatures)

In [12]:
import random

# random vectors of +1 and -1 of dimensione FEATURES_SIZE
random_hyperplanes = [[random.choice([-1, 1]) for _ in range(FEATURES_SIZE)] for _ in range(LSH_NUM_SKETCHES)]
b_hyperplanes = sc.broadcast(random_hyperplanes)
print(f"Generated {LSH_NUM_SKETCHES} random hyperplanes (broadcasting ~{(LSH_NUM_SKETCHES * FEATURES_SIZE * 4) // 1024} KB)")

def compute_sketch(row):
    item_id, sparse_features = row
    sketch = []
    for plane in b_hyperplanes.value:
        # dot product: if above or below the hyperplane
        # only considering the features that exist in the sparse vector
        dot_product = sum(weight * plane[feat] for feat, weight in sparse_features)
        # 1 if positive, 0 if negative
        sketch.append(1 if dot_product > 0 else 0)
    return (item_id, sketch)

# compute sketches ("equivalent" of signature) for users and articles
users_sketches = (users_profile # (user_id, (feature_index, frequency))
                  .groupByKey()
                  .mapValues(list) # (user_id, [(feature_index, frequency), ...])
                  .map(compute_sketch)) # (user_id, sketch)
articles_sketches = (articles_profile # (article_id, (feature_index, frequency = 1))
                     .groupByKey()
                     .mapValues(list) # (article_id, [(feature_index, frequency), ...])
                     .map(compute_sketch)) # (article_id, sketch)

print(f"Number of users sketches: {users_sketches.count()}")
print(f"Number of articles sketches: {articles_sketches.count()}")
users_sketches.take(1), articles_sketches.take(1)


Generated 100 random hyperplanes (broadcasting ~1953 KB)
Number of users sketches: 91437
Number of articles sketches: 16787


([(13120442,
   [0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    1,
    1,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    1,
    0,
    0,
    1])],
 [('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd',
   [1,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
   

## LSH buckets

In [13]:
def lsh_buckets(row):
    item_id, sketch = row
    res = []
    for band in range(LSH_BANDS):
        band_signature = tuple(sketch[band*LSH_ROWS : (band+1)*LSH_ROWS])
        bucket_hash = h(f"b{band}_{band_signature}") # include band index to avoid collisions between different bands
        res.append((bucket_hash, item_id)) # no need to differentiate between users and articles: different format and distinct rdds
    return res

users_buckets = users_sketches.flatMap(lsh_buckets) # (bucket_hash, user_id)
articles_buckets = articles_sketches.flatMap(lsh_buckets) # (bucket_hash, article_id)

print(f"Total number of users buckets: {users_buckets.count()}")
print(f"Total number of articles buckets: {articles_buckets.count()}")

# buckets too big are not useful, too few information
valid_users_buckets = (users_buckets
                       .map(lambda x: (x[0], 1))
                       .reduceByKey(lambda a, b: a + b)
                       .filter(lambda x: x[1] <= LSH_MAX_BUCKET_SIZE))

valid_articles_buckets = (articles_buckets
                          .map(lambda x: (x[0], 1))
                          .reduceByKey(lambda a, b: a + b)
                          .filter(lambda x: x[1] <= LSH_MAX_BUCKET_SIZE))

# the join works as an intersection between keys
users_buckets = (users_buckets # (bucket_hash, user_id)
                 .join(valid_users_buckets) # (bucket_hash, (user_id, count))
                 .map(lambda x: (x[0], x[1][0]))) # (bucket_hash, user_id)
articles_buckets = (articles_buckets # (bucket_hash, article_id)
                    .join(valid_articles_buckets) # (bucket_hash, (article_id, count))
                    .map(lambda x: (x[0], x[1][0]))) # (bucket_hash, article_id)

print(f"Valid (less than {LSH_MAX_BUCKET_SIZE} users) number of users buckets: {users_buckets.count()} ")
print(f"Valid (less than {LSH_MAX_BUCKET_SIZE} articles) number of articles buckets: {articles_buckets.count()} ")

users_buckets.take(10), articles_buckets.take(10)


Total number of users buckets: 914370
Total number of articles buckets: 167870
Valid (less than 500 users) number of users buckets: 535084 
Valid (less than 500 articles) number of articles buckets: 160998 


([(-274404184, 79217680),
  (-274404184, 72890248),
  (-274404184, 71140410),
  (-274404184, 63561257),
  (1936691908, 79217680),
  (1936691908, 72890248),
  (1936691908, 56818706),
  (1936691908, 83095326),
  (1936691908, 66911752),
  (1936691908, 91192218)],
 [(635453318, 'nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd'),
  (635453318, 'nyt://article/af2afd2c-d37d-5a32-b068-234b3d051261'),
  (635453318, 'nyt://article/1db0db41-3cb8-5729-8578-2a10871f3f31'),
  (635453318, 'nyt://article/e530651d-4bbd-59e9-82db-9e0461b3acd5'),
  (635453318, 'nyt://article/5ccbb614-bb92-505e-a38f-99ce0320f0df'),
  (635453318, 'nyt://article/1e0bdb9e-b2c4-5afd-ba42-b73ed2fe86c5'),
  (635453318, 'nyt://article/70dad876-d175-51d2-9ce6-2b2476ea0085'),
  (635453318, 'nyt://article/0676e971-3475-5df1-ab1c-f5fa22e5a9d0'),
  (635453318, 'nyt://article/a1274561-1276-54f9-a657-fa192e810c3e'),
  (635453318, 'nyt://article/7e757913-14e9-5c04-81e2-e540bf31768c')])

## LSH filter

In [14]:
#           BUCKET
# APPROACH  PRUNING  DISTINCT  SUBTRACT  BANDS  ROWS  RESULTS  TIME
#   hash      no        no       no       20     5     ~1.3b  3min 20s
#   hash      no        no       no       5      20    ~13m        13s
#   hash      no        no       no       10     10    ~60m        20s
#   hash      no       yes       no       10     10    ~35m   5min 40s
#   hash     yes        no       no       10     10    ~13m         2s
#   hash     yes       yes       no       10     10    ~12m        20s
#   hash     yes       yes      yes       10     10    ~12m   4min

lsh_recommendations = (users_buckets # (bucket_hash, user_id)
                       .join(articles_buckets) # (bucket_hash, (user_id, article_id))
                       .map(lambda x: (x[1][0], x[1][1])) # (user_id, article_id)
                       .distinct() # remove matches from multiple bands
                       .subtract(utility_matrix.map(lambda x: (x[1], x[0])))) # remove already commented articles

print(f"Number of recommendations that passed LSH filter: {lsh_recommendations.count()}")
lsh_recommendations.take(10)


Number of recommendations that passed LSH filter: 12540194


[(88474684, 'nyt://article/455275d1-895b-5421-9355-8e70e4b90449'),
 (31906798, 'nyt://article/ec98c69d-5ec8-5a2b-a2c1-741b49924b7d'),
 (83896394, 'nyt://article/7465b3eb-6797-5959-a288-def5f8d3580d'),
 (96146747, 'nyt://interactive/18020931-f958-5cea-92bb-2928645e0c4c'),
 (67857496, 'nyt://article/10c8366f-fbd3-5a71-bd2c-51e6ea47aecf'),
 (44444258, 'nyt://article/8cfbaf05-4a2d-5b3e-bc64-3708be964011'),
 (105516007, 'nyt://article/23dfa8a7-832a-589f-9085-4848c1aff8d5'),
 (46840743, 'nyt://article/ed5b900c-9ec0-56c6-b333-8a711baa066d'),
 (61057859, 'nyt://article/9ad4f504-279f-5595-9f59-6e3c25059e7e'),
 (27363762, 'nyt://article/76191623-1f3b-51c4-a9a1-11c084d537a4')]

## Actual Cosine Similarity

In [15]:
# cosine similarity between sparse profiles, user_profile and article_profile should be dicts {feature_index: weight}
def cosine_similarity(user_profile, article_profile):
    dot_product = sum(user_profile.get(feat, 0) * weight for feat, weight in article_profile.items())
    user_norm = sum(weight ** 2 for weight in user_profile.values()) ** 0.5
    article_norm = sum(weight ** 2 for weight in article_profile.values()) ** 0.5
    if user_norm == 0 or article_norm == 0:
        return 0.0
    return dot_product / (user_norm * article_norm)

# calculate actual similarity by joining the profiles and applying
recommendations = (lsh_recommendations # (user_id, article_id)
                   .join(users_profile.groupByKey().mapValues(dict)) # (user_id, (article_id, {profile}))
                   .map(lambda x: (x[1][0], (x[0], x[1][1]))) # (article_id, (user_id, {user_profile}))
                   .join(articles_profile.groupByKey().mapValues(dict)) # (article_id, ((user_id, {user_profile}), {article_profile}))
                   .map(lambda x: (x[1][0][0], (x[0], cosine_similarity(x[1][0][1], x[1][1])))) # (user_id, (article_id, actual_sim))
                   .filter(lambda x: x[1][1] > 0.0))

print(f"Number of recommendations with actual cosine similarity: {recommendations.count()}")
recommendations.take(10)


Number of recommendations with actual cosine similarity: 6276258


[(87995772,
  ('nyt://article/92f6fb86-d51b-50a6-b251-7bc85e8cb00d', 0.07009996372327816)),
 (81687429,
  ('nyt://article/92f6fb86-d51b-50a6-b251-7bc85e8cb00d', 0.08362420100070908)),
 (74566062,
  ('nyt://article/92f6fb86-d51b-50a6-b251-7bc85e8cb00d', 0.20100756305184242)),
 (23161455,
  ('nyt://article/92f6fb86-d51b-50a6-b251-7bc85e8cb00d', 0.02890903309108255)),
 (54054279,
  ('nyt://article/92f6fb86-d51b-50a6-b251-7bc85e8cb00d', 0.08022494521202195)),
 (538209,
  ('nyt://article/92f6fb86-d51b-50a6-b251-7bc85e8cb00d', 0.08362420100070908)),
 (68628231,
  ('nyt://article/92f6fb86-d51b-50a6-b251-7bc85e8cb00d',
   0.055048188256318034)),
 (89778924,
  ('nyt://article/92f6fb86-d51b-50a6-b251-7bc85e8cb00d', 0.08362420100070908)),
 (69447762,
  ('nyt://article/92f6fb86-d51b-50a6-b251-7bc85e8cb00d',
   0.017554669175502146)),
 (18189342,
  ('nyt://article/92f6fb86-d51b-50a6-b251-7bc85e8cb00d',
   0.019837990021453852))]

# Content-Based Recommendations

In [16]:
import heapq

user_recommendations = (recommendations
                        .map(lambda x: (x[0], (x[1][0], round(x[1][1], 5))))
                        .groupByKey()
                        .mapValues(lambda iter: heapq.nlargest(RECOMMENDED_ARTICLES, iter, key=lambda x: x[1])))

print(f"Number of users with at least one recommendation: {user_recommendations.count()}/{users.count()}")
user_recommendations.take(10)


Number of users with at least one recommendation: 82875/91437


[(69645510,
  [('nyt://article/ac9d08a8-e77e-5ddb-be11-307f9a2108e3', 0.66225),
   ('nyt://article/cb728867-5208-5eb5-bc27-ed4c7313ae27', 0.59378),
   ('nyt://article/9bb1f6dc-b99d-5b22-814b-92ab740ac5d7', 0.57735),
   ('nyt://article/f554c8a5-c4cb-5d5c-8c3f-8f1d5b24cdaa', 0.57735),
   ('nyt://article/3e04f020-aff1-524c-a9ba-edf82192403c', 0.56679),
   ('nyt://article/cc543364-4295-5f61-a40c-d62000bae11e', 0.55024),
   ('nyt://article/b7f57483-8051-5074-bed5-a4010b633ebf', 0.54412),
   ('nyt://article/7d702034-7910-5ea3-94a7-c0f03ac8a296', 0.53945),
   ('nyt://article/cdfe4b75-ead2-5aba-ae5e-8986bc4ab37b', 0.51993),
   ('nyt://article/6ba2e6d3-f4e0-59df-b6ff-5fa8501174c8', 0.50943)]),
 (87098040,
  [('nyt://article/0457a10e-6ae9-584f-aa93-e49474764a7e', 0.5),
   ('nyt://article/483f1ed1-0027-5c15-b3f5-66632346adef', 0.43301),
   ('nyt://article/d4b0af48-3593-5010-802d-c1b1069c8f15', 0.40089),
   ('nyt://article/c6ab0e5d-c76d-5da3-8c17-68cb6cd57feb', 0.375),
   ('nyt://article/dced2ad6-

## Human Evaluation

In [45]:
USER = users.takeSample(False, 1)[0]

# "notable" users:
# USER = 65969500 # bad results
# USER = 22564580 # policits but not trump

read = (utility_matrix
        .filter(lambda x: x[1] == USER)
        .join(full_articles.rdd.map(lambda x: (x['uniqueID'], x.asDict()))) # (article_id, (user_id, article_info))
        .map(lambda x: x[1][1]) # (article_info)
        .collect())

reccs = (user_recommendations
         .filter(lambda x: x[0] == USER)
         .map(lambda x: x[1])
         .flatMap(lambda x: x)
         .join(full_articles.rdd.map(lambda x: (x['uniqueID'], x.asDict()))) # (article_id, article_info)
         .map(lambda x: (x[1][1], x[1][0])) # (article_info, similarity)
         .collect())

print("--- ARTICLES COMMENTED ---")
for r in read:
    kws = "\n\t".join(parse_keywords(r))
    print(f"  {r['headline']}\n\t{r['newsdesk']} - {r['section']} - {r['subsection']}\n\t{kws}")

print("--- ARTICLES RECOMMENDED ---")
for r, s in sorted(reccs, key=lambda x: x[1], reverse=True):
    kws = "\n\t".join(parse_keywords(r))
    print(f"  {round(s, 5)} {r['headline']}\n\t{r['newsdesk']} - {r['section']} - {r['subsection']}\n\t{kws}")


--- ARTICLES COMMENTED ---
  Trump the Intimidator Fails Again
	OpEd - Opinion - None
	united states international relations
	international trade and world market
	united states defense and military forces
	united states politics and government
	suleimani
	qassim
	trump
	donald j
	iran
  Bezos Phone Hack Tied to Saudi Crown Prince Puts New Pressure on Kingdom
	Foreign - World - Middle East
	united nations
	amazon.com inc
	bezos
	jeffrey p
	mohammed bin salman (1985- )
	khashoggi
	jamal
	cyberattacks and hackers
--- ARTICLES RECOMMENDED ---
  0.44998 Did Trump Do the Right Thing with Iran?
	OpEd - Opinion - None
	iran
	suleimani
	qassim
	drones (pilotless planes)
	obama
	barack
	assassinations and attempted assassinations
	trump
	donald j
  0.34188 How Trump Lowered America’s Standing in the World
	OpEd - Opinion - None
	trump
	donald j
	united states international relations
	united states politics and government
	nationalism (theory and philosophy)
	european union
	europe
	china
	russi